In [0]:
from pyspark.sql.functions import (
    col, to_date, regexp_replace, year, when, trim, split, explode, desc, avg, count
)
from pyspark.sql.types import DoubleType, IntegerType

df_bronze = spark.table("workspace.default.imdb_tmdb_movie_metadata_big_dataset_1_m")

df_silver = (
    df_bronze
    # 1) Typage explicite (Spark infère souvent en string)
    .withColumn("budget",        col("budget").cast(DoubleType()))
    .withColumn("revenue",       col("revenue").cast(DoubleType()))
    .withColumn("vote_average",  col("vote_average").cast(DoubleType()))
    .withColumn("vote_count",    col("vote_count").cast(IntegerType()))
    .withColumn("runtime",       col("runtime").cast(IntegerType()))
    .withColumn("popularity",    col("popularity").cast(DoubleType()))
    # 2) Les 0 dans budget/revenue signifient "manquant" → NULL
    .withColumn("budget",  when(col("budget")  <= 0, None).otherwise(col("budget")))
    .withColumn("revenue", when(col("revenue") <= 0, None).otherwise(col("revenue")))
    # 3) Dates normalisées (colonne déjà typée DATE, rendue au format ISO "yyyy-MM-dd")
    .withColumn("release_date_clean",
                to_date(regexp_replace(col("release_date"), "/", "-"), "yyyy-MM-dd"))
    .withColumn("annee",    year("release_date_clean"))
    .withColumn("decennie", (col("annee") / 10).cast("int") * 10)
    # 4) Colonnes dérivées métier
    .withColumn("roi",
                when(col("budget").isNotNull() & col("revenue").isNotNull(),
                     col("revenue") / col("budget")))
    .withColumn("profit",
                col("revenue") - col("budget"))
    # 5) Nettoyage texte
    .withColumn("title", trim(col("title")))
    # 6) Dédoublonnage sur la clé primaire
    .dropDuplicates(["id"])
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.default.films_silver")

print(f"Silver : {spark.table('workspace.default.films_silver').count():,} films")